# Example usage of careful-scada
from data loading to building models.

## Data Loading & Preprocessing

In [ ]:
import sys, os
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go

sys.path.append(os.path.abspath(".."))
# add src folder to path
sys.path.append(os.path.abspath("../care_scada"))
print(sys.path)

from care_scada.data_loader import DataLoader
from care_scada.preprocessing import Preprocessor
from care_scada.plotter import plot_sensor_timeseries
from care_scada.data_factory import DatasetFactory
from care_scada.model_factory import ModelFactory

### Dataloading & Windfarm selection
The CARE_To_Compare dataset contains three windfarms with each containing multiple datasets itself. See the README.md from the 
authors for CARE_To_Compare for more details.
Lets load on of the farms:

Note how the loader prints the number of currently supported features. Also the number of loaded datesets from the farm is printed.

In [ ]:
farm_name = 'B'  # or 'B', 'C'
loader = DataLoader(farm_name=farm_name, data_path="dat/Care_To_Compare")  # relative to the repo root  # or 'B', 'C'
df = loader.load()

### Preprocessing & Dataset selection
The loaded data is now preprocessed according to the specifications and made ready for modelling.
Note how the Preprocessor offers different strategies for e.g. the handling of nan values. Also
it is up to you, if you want to drop oultiers, or only flag them as such.

For a faster experience, set an dataset id (ds_id) and only preprocess that specific dataset from the farm.

In [ ]:
ds_id = 77
pre = Preprocessor(
    farm_name=farm_name,
    dataset_id=ds_id,
    missing_values="drop",
    outlier_handling="drop",
)
df_train_clean, df_test_clean = pre.process(df)

Great, lets have a look at one of the datasets loaded. 

In [ ]:
plot_sensor_timeseries(
    df=df_train_clean, 
    dataset_id='77', 
    sensor_cols=pre.numeric_cols,
    df_test=df_test_clean,
    #show_power=True,
    show_outliers=True) 

# Modelling
Now the data is ready for modelling. Lets pass it in the DatasetFactory, that creates dataset_id level
dataframes for training.

In [ ]:
datasets = DatasetFactory.create(
    farm_name='B',
    train_df=df_train_clean,
    test_df=df_test_clean
)
# get the first dataset
ds = datasets[0]
print(ds.farm_name, ds.dataset_id, ds.train_df.shape, ds.test_df.shape)

Now the actual modelling starts: 
In this example, the target_col defines the target of six
different regression models, that are created using the ModelFactory. 
This setup allows to train models on all datasets of the the farm. 

In [ ]:
results = []
trained_models = {}
target_col = 'main_bearing_temp2_c_avg'

for ds in datasets:
    models = []
    for model_name in ['naive_forecaster', 'constant_forecaster', 'random_forest', 'lstm', 'pcrnn', 'kalman']:
        model = ModelFactory.create(
            model_name=model_name)
        model.fit(ds.train_df, target_col=target_col)
        predictions = model.predict(ds.test_df, target_col=target_col)
        rmse = root_mean_squared_error(ds.test_df[target_col], predictions[target_col])

        results.append({
            'farm_name': ds.farm_name,
            'dataset_id': ds.dataset_id,
            'model_name': model_name,
            'rmse': rmse
        })
        # Store model for later access
        key = (ds.farm_name, ds.dataset_id, model_name)
        trained_models[key] = model

Let's aggregate the resulting RMSE for the six implemented models and visualize the regression.

In [ ]:
# barplot, average RMSE per model
results_df = pd.DataFrame(results)
avg_rmse = results_df.groupby('model_name')['rmse'].mean()
avg_rmse.plot(kind='bar')
plt.ylabel('Average RMSE')
plt.title('Average RMSE per Model')
plt.show()

In [ ]:
ds = datasets[0]  # Visualize for the first dataset


fig = go.Figure()

models_list = ['naive_forecaster', 'constant_forecaster', 'random_forest', 'lstm', 'pcrnn', 'kalman']

# Traindata
fig.add_trace(go.Scatter(x=ds.train_df.index, y=ds.train_df[target_col],
                            mode='lines', name='Train Actual', 
                            line=dict(color='black', width=1),
                            legendgroup='actual'))
# Testdata
fig.add_trace(go.Scatter(x=ds.test_df.index, y=ds.test_df[target_col],
                            mode='lines', name='Test Actual', 
                            line=dict(color='gray', width=1),
                            legendgroup='actual'))

# Predictions
for model_name in models_list:
    key = (ds.farm_name, ds.dataset_id, model_name)
    model = trained_models.get(key)
    
    if model:
        train_preds = model.predict(ds.train_df, target_col=target_col)
        test_preds = model.predict(ds.test_df, target_col=target_col)

        # Train Prediction Trace
        fig.add_trace(go.Scatter(x=ds.train_df.index, y=train_preds[target_col],
                                    mode='lines', name=f'{model_name} (Train)',
                                    line=dict(dash='dash'),
                                    visible=False)) 
        # Test Prediction Trace
        fig.add_trace(go.Scatter(x=ds.test_df.index, y=test_preds[target_col],
                                    mode='lines', name=f'{model_name} (Test)',
                                    line=dict(dash='dash'),
                                    visible=False))

# Add Dropdown Buttons for Model Selection
buttons = []
start_idx_predictions = 2 # Index where predictions start (after the 2 Actual Traces)
traces_per_model = 2 # Train + Test Prediction per model

# Initialization: Make the first model visible
if len(fig.data) > start_idx_predictions:
    fig.data[start_idx_predictions].visible = True
    fig.data[start_idx_predictions+1].visible = True

for i, model_name in enumerate(models_list):
    # Create visibility mask: 
    # [True, True] (Actuals) + [False...] (all preds)
    visibility = [True, True] + [False] * (len(models_list) * traces_per_model)
    
    # Activate visibility for current model
    curr_idx = start_idx_predictions + (i * traces_per_model)
    # Ensure we don't go out-of-bounds (in case models are missing)
    if curr_idx + 1 < len(visibility):
        visibility[curr_idx] = True
        visibility[curr_idx + 1] = True
    
    button = dict(
        label=model_name,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Actual vs Predicted: {model_name}"}]
    )
    buttons.append(button)

# Layout Update with Menu
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=1.15, 
        xanchor="left",
        y=1.15,
        yanchor="top"
    )],
    title=f"Actual vs Predicted: {models_list[0]}",
    xaxis_title="Time",
    yaxis_title=target_col,
    template="plotly_white"
)

The physics-constrained RNN (`pcrnn`) is built on a lumped thermal network: the bearing temperature
exchanges heat with neighbouring measured temperatures and is heated by operating signals, all
with learned coefficients. The plot shows the network with the learned conductances, time constants and
temperature rises at full load.

### Defining a heat network
The `pcrnn` and `kalman` models and the `kalman` detector share one network description, the
`ThermalNetworkSpec`:

* `target`: the predicted temperature,
* `neighbours`: measured temperatures that enter the differential equation of the target via the temperature diffference,
* `sources`: measured non-temperature signals that enter the differential equation of the target as purely positive (!) terms,
either linearily or quadratically `{column: 'linear' | 'square'}`. Use
  `{column: (node, transform)}` to heat another state,
* `latent`: unmeasured nodes with their own heat balance (e.g. the nacelle air),
* `measured_states`: measured temperatures with their own heat balance. `with_measured_states`
  couples them to the target and gives them the target's neighbours and sources.

Connect `latent` nodes and `measured_states` nodes to other nodes via `edges`.

When no custom network is given, the models default to the network configuration in `sensor_config.yaml`.
Override single entries by adding keywords arguments with the same name (e.g. `ModelFactory.create('kalman', measured_states=[...])`) or define custom network.
If a measured variable does not exist in the dataset, it is dropped.

The spec only describes the physics. A model *predicts* the spec's `target`; a detector *monitors*
one or more temperatures of the network (`monitored`, see Fault detection).

Attention! While the Kalman model splits the target into a latent node and an observed noisy measurement node, the PCRNN has no observation model and treats thus the observation of the target as the exact target value.


#### Default thermal network (PCRNN)

In [ ]:
from care_scada.plotter import plot_thermal_network

plot_thermal_network(trained_models[(ds.farm_name, ds.dataset_id, 'pcrnn')]).show()

#### Custom Thermal Network

In [ ]:
from care_scada.modeling.thermal_network import ThermalNetworkSpec

# Define a thermal network specification for the target variable and plot the network structure
network = ThermalNetworkSpec(
    target=target_col,
    neighbours=['outside_temp_c_avg', 'gearbox_bearing_temp1_c_avg'],
    sources={'rotor_speed_rpm_avg': 'linear', 'active_power_kWh': 'linear'},
).with_measured_states(['main_bearing_temp1_c_avg'])
plot_thermal_network(network).show()

# Plot the Kalman filter model fitted to the dataset
kalman = ModelFactory.create('kalman', network=network).fit(ds.train_df, target_col)
plot_thermal_network(kalman).show()


# Fault detection
A **detector** computes one daily indicator per **monitored** temperature and raises an alarm when an
indicator changes unusually. It works in three steps:

1. `fit`: fit the detector's model on healthy data,
2. `calibrate`: compute the indicators of healthy turbines; the alarm thresholds are the largest
   values the alarm checks reach on them,
3. `detect`: list the alarms (day, monitored temperature, kind) of one turbine.

There are two alarm checks (CUSUMs) per monitored temperature: `rise` fires on a fast daily
increase, `level` on an offset from the indicator's median of the preceding months.

Details: 
`CUMSUM` $S_d = max(0, S_{d-1} + \text{inc}_d - k)$ with $k$ being a small tolerance parameter

`rise`: $inc_d = x_d - x_{d_1}$

`level`: $inc_d = x_d - \text{baseline}_d$ with $\text{baseline}$ being the median of $d-90$ to $d-30$.



**Target vs monitored.** A model *predicts* one temperature, the network's `target`. A detector
*monitors* one or more temperatures (`monitored`). Created from `network` (above), the `kalman`
detector monitors the network's target and all its measured states, here both main bearings.

**Why a Kalman detector?** A residual detector (`DetectorFactory.create("residual", model_name="pcrnn",
monitored=[...])`) monitors the measured minus the predicted temperature. But the PCRNN predicts
bearing 2 from bearing 1: if bearing 1 heats up, the prediction of bearing 2 rises with it, and part
of the fault hides in the residual. The Kalman detector models both bearings as states with noisy
sensors and tracks a constant heat term for each. A fault adds heat, so the tracked steady-state
excess temperature of the faulty bearing rises, and the alarm names that bearing.

**Setup.** The physics is fitted on healthy data (dataset 83, and dataset 86 before its bearing-2
drift) and recalibrated on the first 90 days of each turbine, so alarms are possible only after day 90.
Datasets 83, 52, 21, 2, 74, 82 are healthy turbines and set the thresholds; 77, 53 and 27 have
main-bearing damage. Runtime is a few minutes.


In [ ]:
from care_scada.detector_factory import DetectorFactory

def turbine_series(ds_id):
    """Train and test part of one dataset as a single time-indexed series."""
    pre = Preprocessor(farm_name='B', dataset_id=ds_id, missing_values="drop",
                       outlier_handling="drop", verbose=False)
    return pd.concat(pre.process(df)).droplevel(0).sort_index()

healthy_ids, fault_ids = [83, 52, 21, 2, 74, 82], [77, 53, 27]  # fault_ids: main-bearing damage
series = {d: turbine_series(d) for d in healthy_ids + fault_ids + [86]}
# monitors the network's target and measured states (same as monitored=[bearing 2, bearing 1])
detector = DetectorFactory.create("kalman", network=network)
bearings = detector.monitored
print("monitored:", bearings)
detector.fit(pd.concat([series[83], series[86][:"2022-09-30"]]))
indicators = {d: detector.indicators(series[d]) for d in healthy_ids + fault_ids}
print("thresholds:", detector.calibrate([indicators[d] for d in healthy_ids]))


The detector's heat network is the same `network`, but now both bearings are monitored (red). Both are states, measured with noise and coupled to each other. Each
bearing exchanges heat with the outside temperature and the gearbox bearing, and is heated by rotor speed and power.

In [ ]:
plot_thermal_network(detector).show()

In [ ]:
events = pd.read_csv(os.path.join(loader.path, "..", "event_info.csv"), sep=";").set_index("event_id")

for d in fault_ids:
    alarms = detector.detect(indicators[d])
    start = pd.Timestamp(events.loc[d, "event_start"])
    first = alarms["day"].iloc[0] if len(alarms) else None
    print(f"dataset {d}: event {start.date()}, first alarm {first.date() if first else None}")

d = 77
ind, alarms = indicators[d], detector.detect(indicators[d])
event = events.loc[d]
alarms["days_after_event_start"] = (alarms["day"] - pd.Timestamp(event["event_start"])).dt.days
print(f"CARE fault label: {event['event_start']} – {event['event_end']}, '{event['event_description']}'")
print(alarms)

# colour = bearing that heats up, marker = CUSUM that fired (rise: fast increase, level: offset)
colors = dict(zip(bearings, ['#E4572E', '#4C78A8']))
markers = {'rise': ('triangle-up', 'dot'), 'level': ('diamond', 'dash')}
fig = go.Figure()
for b in bearings:
    fig.add_trace(go.Scatter(x=ind.index, y=ind[b], mode='lines', name=b, line=dict(color=colors[b])))
# grey: anomaly event labelled in CARE's event_info.csv
fig.add_vrect(x0=event["event_start"], x1=event["event_end"], fillcolor='gray', opacity=0.2,
              line_width=0, annotation_text='CARE fault label', showlegend=True,
              name=f"CARE fault label ({event['event_start'][:10]} – {event['event_end'][:10]})")
for (b, kind), a in alarms.groupby(['monitored', 'kind']):
    symbol, dash = markers[kind]
    for day in a['day']:
        fig.add_vline(x=day, line_dash=dash, line_width=1, line_color=colors[b])
    fig.add_trace(go.Scatter(x=a['day'], y=ind[b].reindex(a['day']), mode='markers',
                             name=f'alarm: {b} ({kind})',
                             marker=dict(symbol=symbol, size=13, color=colors[b],
                                         line=dict(color='black', width=1))))
fig.update_layout(title=f"Dataset {d}: tracked steady-state excess temperature and alarms",
                  yaxis_title="K", legend=dict(orientation='h', y=-0.15))
fig.show()
